In [ ]:
import numpy as np
from scipy.optimize import linprog
import time


def solve_velocity_planning_lp(
    dt,
    s_hat,
    vmin,
    vmax,
    smin,
    smax,
    amin,
    amax,
    jmin,
    jmax,
    s0=None,
    v0=None,
):
    """
    Solve the velocity planning problem as a linear program.

    Variables:
        x = [s_0, ..., s_{T-1}, v_0, ..., v_{T-1}]

    Objective:
        minimize sum_t (s_hat[t] - s[t])
        equivalent to minimize -sum_t s[t]

    Constraints:
        1) s_{t+1} = s_t + 0.5*(v_t + v_{t+1})*dt
        2) vmin_t <= v_t <= vmax_t
        3) amin <= (v_{t+1} - v_t)/dt <= amax
        4) jmin <= (v_{t+2} + v_t - 2*v_{t+1})/dt^2 <= jmax
        5) smin_t <= s_t <= smax_t, and s_t <= s_hat_t

    Optional:
        s0: if not None, enforce s_0 == s0
        v0: if not None, enforce v_0 == v0
    """
    s_hat = np.asarray(s_hat, dtype=float)
    vmin = np.asarray(vmin, dtype=float)
    vmax = np.asarray(vmax, dtype=float)
    smin = np.asarray(smin, dtype=float)
    smax = np.asarray(smax, dtype=float)

    T = len(s_hat)
    assert len(vmin) == T
    assert len(vmax) == T
    assert len(smin) == T
    assert len(smax) == T
    assert dt > 0.0

    n_s = T
    n_v = T
    n_x = n_s + n_v

    def s_idx(t):
        return t

    def v_idx(t):
        return n_s + t

    # ---------------------------
    # Objective: minimize -sum(s_t)
    # ---------------------------
    c = np.zeros(n_x)
    for t in range(T):
        c[s_idx(t)] = -1.0

    # ---------------------------
    # Equality constraints
    # ---------------------------
    A_eq = []
    b_eq = []

    # 1) Dynamics:
    # s_{t+1} - s_t - 0.5*dt*v_t - 0.5*dt*v_{t+1} = 0
    for t in range(T - 1):
        row = np.zeros(n_x)
        row[s_idx(t + 1)] = 1.0
        row[s_idx(t)] = -1.0
        row[v_idx(t)] = -0.5 * dt
        row[v_idx(t + 1)] = -0.5 * dt
        A_eq.append(row)
        b_eq.append(0.0)

    # Optional fixed initial position
    if s0 is not None:
        row = np.zeros(n_x)
        row[s_idx(0)] = 1.0
        A_eq.append(row)
        b_eq.append(float(s0))

    # Optional fixed initial velocity
    if v0 is not None:
        row = np.zeros(n_x)
        row[v_idx(0)] = 1.0
        A_eq.append(row)
        b_eq.append(float(v0))

    A_eq = np.array(A_eq) if A_eq else None
    b_eq = np.array(b_eq) if b_eq else None

    # ---------------------------
    # Inequality constraints A_ub x <= b_ub
    # ---------------------------
    A_ub = []
    b_ub = []

    # 3) Acceleration constraints
    # v_{t+1} - v_t <= amax * dt
    # v_{t+1} - v_t >= amin * dt
    # -> -(v_{t+1} - v_t) <= -amin*dt
    for t in range(T - 1):
        row = np.zeros(n_x)
        row[v_idx(t + 1)] = 1.0
        row[v_idx(t)] = -1.0
        A_ub.append(row)
        b_ub.append(amax * dt)

        row = np.zeros(n_x)
        row[v_idx(t + 1)] = -1.0
        row[v_idx(t)] = 1.0
        A_ub.append(row)
        b_ub.append(-amin * dt)

    # 4) Jerk constraints
    # v_{t+2} + v_t - 2*v_{t+1} <= jmax * dt^2
    # v_{t+2} + v_t - 2*v_{t+1} >= jmin * dt^2
    # -> -(v_{t+2} + v_t - 2*v_{t+1}) <= -jmin * dt^2
    for t in range(T - 2):
        row = np.zeros(n_x)
        row[v_idx(t)] = 1.0
        row[v_idx(t + 1)] = -2.0
        row[v_idx(t + 2)] = 1.0
        A_ub.append(row)
        b_ub.append(jmax * dt * dt)

        row = np.zeros(n_x)
        row[v_idx(t)] = -1.0
        row[v_idx(t + 1)] = 2.0
        row[v_idx(t + 2)] = -1.0
        A_ub.append(row)
        b_ub.append(-jmin * dt * dt)

    A_ub = np.array(A_ub) if A_ub else None
    b_ub = np.array(b_ub) if b_ub else None

    # ---------------------------
    # Variable bounds
    # ---------------------------
    bounds = []

    # s_t bounds: smin_t <= s_t <= min(smax_t, s_hat_t)
    for t in range(T):
        ub = min(smax[t], s_hat[t])
        lb = smin[t]
        if lb > ub:
            raise ValueError(
                f"Infeasible position bounds at t={t}: "
                f"smin={lb}, min(smax, s_hat)={ub}"
            )
        bounds.append((lb, ub))

    # v_t bounds: vmin_t <= v_t <= vmax_t
    for t in range(T):
        lb = vmin[t]
        ub = vmax[t]
        if lb > ub:
            raise ValueError(
                f"Infeasible velocity bounds at t={t}: vmin={lb}, vmax={ub}"
            )
        bounds.append((lb, ub))

    # ---------------------------
    # Solve LP
    # ---------------------------
    result = linprog(
        c=c,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs",
    )

    if not result.success:
        raise RuntimeError(f"LP failed: {result.message}")

    x = result.x
    s = x[:T]
    v = x[T:]

    return {
        "s": s,
        "v": v,
        "objective_min_sum_s_hat_minus_s": np.sum(s_hat - s),
        "raw_result": result,
    }



# Example usage
T = 20
dt = 0.2

# Reference position hat{s_t}
s_hat = np.linspace(0.0, 20.0, T)

# Velocity bounds
vmin = np.zeros(T)
vmax = 5.0 * np.ones(T)

# Position bounds
smin = np.zeros(T)
smax = s_hat + 2.0   # loose upper bound, actual effective upper bound is min(smax, s_hat)=s_hat

# Global acceleration / jerk bounds
amin = -2.0
amax = 1.5
jmin = -3.0
jmax = 3.0

# Optional initial state
s0 = 0.0
v0 = 0.0

time0 = time.time()
sol = solve_velocity_planning_lp(
    dt=dt,
    s_hat=s_hat,
    vmin=vmin,
    vmax=vmax,
    smin=smin,
    smax=smax,
    amin=amin,
    amax=amax,
    jmin=jmin,
    jmax=jmax,
    s0=s0,
    v0=v0,
)
time1 = time.time()

s = sol["s"]
v = sol["v"]

print("Solved successfully.")
print("s =", np.round(s, 4))
print("v =", np.round(v, 4))
print("objective =", sol["objective_min_sum_s_hat_minus_s"])
print("Time taken =", time1 - time0, "seconds")

Time taken = 0.001531362533569336 seconds
